# 09. AI Compilers and Graph Optimization | AI 编译器与计算图优化

**难度：** Hard | **环境：** CPU-first | **标签：** `编译系统`, `AI Compiler`, `Graph Optimization` | **目标人群：** 编译与系统入门者

![计算图到后端执行路径](../docs/public/01_Hardware_Math_and_Systems/09_compiler_graph_lowering.svg)

---

## 本节导读

模型从高层图走到真正可执行的后端代码，中间并不是简单翻译，而是一整条图变换、算子改写、布局调整和执行计划选择的链路。很多性能差距，并不是来自公式本身，而是来自这条链路有没有把计算改写成更适合目标硬件的形式。

这一页在整个教程的纵向主线里属于 `Part 01` 的编译与执行基础页，优先服务后续 `编译与图优化` 这条基础支撑线，也给 `推理优化路线` 补一层 backend 视角。学完这里，后面再看 `19 / 30 / 32` 以及 `Part 03 / Part 04` 的 kernel 与系统优化页时，你会更容易把“模型能跑”与“模型能高效跑”区分开来；如果这里没学明白，后面通常会把 fusion、lowering、layout 和 backend 约束都当成部署黑盒，难以解释为什么同一模型在不同执行栈上表现差这么多。按专题归类，这一页主要属于 `编译与图优化专题`，并和 `推理优化专题` 共享一部分执行链路视角。

**关键词：** `AI Compiler`, `Graph Optimization`, `fusion`

---

## 前置阅读
**导语：** 这一页先把图优化、fusion 和 lowering 的关系接上，再理解编译器为什么会成为推理和部署链路里的关键一环。
- [Group 1D: Execution Models, Asynchrony and Operator Mapping | 1D: 执行模型、异步调度与算子映射](./1D.md)
- [Group 1E: Compilers and Backends | 1E: 编译与后端](./1E.md)
- [04. Attention MHA GQA | 多头注意力](../02_PyTorch_Algorithms/04_Attention_MHA_GQA.ipynb)

## Q1：AI 编译器到底在压缩什么成本向量？


AI 编译器优化的对象不是 Python 语法，而是计算图在 lowering 过程中形成的成本向量。

它关心的不是“这段代码长什么样”，而是这张图在落到具体 backend 前会产生多少 launch、多少中间张量、多少布局重写、多少同步点。

所以编译器不是把代码“翻译一下”，而是在同一张图上做代价最小化的执行路径重写。


### Q1 机制计算器：把 launch、物化张量和布局重写换算成成本

先问自己是算子形状、执行顺序，还是中间张量在拖慢系统。

In [ ]:
def graph_cost(num_ops, fused_groups, intermediate_elements, bytes_per_element, layout_bytes=0, launch_overhead_us=5.0):
    """估算 kernel launch 与中间张量读写，不把不同单位相加成任意分数。"""
    if num_ops <= 0 or not 1 <= fused_groups <= num_ops:
        raise ValueError('fused_groups must be in [1, num_ops]')
    if min(intermediate_elements, bytes_per_element, layout_bytes, launch_overhead_us) < 0:
        raise ValueError('sizes and overhead must be non-negative')
    materialized_boundaries = fused_groups - 1
    intermediate_bytes = materialized_boundaries * intermediate_elements * bytes_per_element * 2
    return {
        'launches': fused_groups,
        'launch_overhead_us': fused_groups * launch_overhead_us,
        'materialized_boundaries': materialized_boundaries,
        'intermediate_traffic_bytes': intermediate_bytes,
        'layout_traffic_bytes': layout_bytes,
        'total_extra_traffic_bytes': intermediate_bytes + layout_bytes,
    }

naive = graph_cost(4, 4, 1_000_000, 2)
fused = graph_cost(4, 1, 1_000_000, 2, layout_bytes=1_000_000)
assert fused['launches'] < naive['launches']
assert fused['total_extra_traffic_bytes'] < naive['total_extra_traffic_bytes']
print('naive:', naive)
print('fused:', fused)


## Q2：为什么算子融合必须同时看复用、布局和副作用？


图优化和算子融合之所以经常绑在一起，是因为它们都在处理同一条数据路径：能不能让中间结果不落地、能不能把连续计算收进同一个执行单元、能不能在不破坏语义的前提下复用布局。

当两个算子共享中间张量时，编译器才有融合空间；但是否真能 fuse，还要再看副作用、register pressure 和 layout 是否兼容。

所以融合不是“把两个名字拼起来”，而是同时满足复用、正确性和局部资源约束。


### Q2 硬门槛：融合收益是否覆盖布局与资源代价

如果两个算子共享同一批中间结果，就要优先考虑是否能在图层合并。

In [ ]:
def fusion_candidate(shared_bytes, layout_rewrite_bytes, no_side_effect, register_bytes, register_budget_bytes):
    """先执行语义和资源硬门槛，再计算净减少的外部流量。"""
    if min(shared_bytes, layout_rewrite_bytes, register_bytes, register_budget_bytes) < 0:
        raise ValueError('byte counts must be non-negative')
    reasons = []
    if not no_side_effect:
        reasons.append('side_effect')
    if register_bytes > register_budget_bytes:
        reasons.append('register_budget')
    net_traffic_saved_bytes = shared_bytes - layout_rewrite_bytes
    if net_traffic_saved_bytes <= 0:
        reasons.append('no_net_traffic_saving')
    return {'eligible': not reasons, 'net_traffic_saved_bytes': net_traffic_saved_bytes, 'reasons': reasons}

ideal = fusion_candidate(8_000_000, 0, True, 48_000, 64_000)
layout_bad = fusion_candidate(8_000_000, 10_000_000, True, 48_000, 64_000)
side_effect = fusion_candidate(8_000_000, 0, False, 48_000, 64_000)
assert ideal['eligible'] is True
assert layout_bad['eligible'] is False
assert side_effect['eligible'] is False
print(ideal)
print(layout_bad)
print(side_effect)


## Q3：为什么同一张图在不同 backend 上会得到不同的最优解？


同一张图在不同 backend 上会得到不同的最优解，不是因为图的语义变了，而是因为目标硬件允许的布局、调度和 kernel 形态不同。

编译器在 lowering 的时候，不只是把算子往下放，还要同时满足静态形状、内存布局、算子可用性和性能上限这些约束；一旦某个约束不满足，图优化策略就会变。

所以图优化的最后一步，其实是把“图上的理想解”压回“backend 上可执行的约束解”。


### Q3 约束筛选：先排除不可执行后端，再比较候选延迟

先想 backend 和硬件支持差异，再看图本身。

In [ ]:
def choose_backend(requirements, candidates):
    """使用算子、dtype、动态形状和显存硬约束筛选后端。"""
    eligible = []
    rejected = {}
    for name, candidate in candidates.items():
        reasons = []
        if not requirements['ops'].issubset(candidate['ops']):
            reasons.append('missing_ops')
        if requirements['dtype'] not in candidate['dtypes']:
            reasons.append('unsupported_dtype')
        if requirements['dynamic_shape'] and not candidate['dynamic_shape']:
            reasons.append('dynamic_shape')
        if candidate['workspace_mb'] > requirements['workspace_limit_mb']:
            reasons.append('workspace')
        if reasons:
            rejected[name] = reasons
        else:
            eligible.append(name)
    selected = min(eligible, key=lambda name: candidates[name]['estimated_latency_ms']) if eligible else None
    return {'selected': selected, 'eligible': eligible, 'rejected': rejected}

requirements = {'ops': {'matmul', 'softmax'}, 'dtype': 'bf16', 'dynamic_shape': True, 'workspace_limit_mb': 512}
candidates = {
    'cpu': {'ops': {'matmul', 'softmax'}, 'dtypes': {'fp32', 'bf16'}, 'dynamic_shape': True, 'workspace_mb': 64, 'estimated_latency_ms': 20.0},
    'cuda': {'ops': {'matmul', 'softmax'}, 'dtypes': {'fp16', 'bf16'}, 'dynamic_shape': True, 'workspace_mb': 256, 'estimated_latency_ms': 3.0},
    'static_only': {'ops': {'matmul', 'softmax'}, 'dtypes': {'bf16'}, 'dynamic_shape': False, 'workspace_mb': 128, 'estimated_latency_ms': 1.5},
}
decision = choose_backend(requirements, candidates)
assert decision['selected'] == 'cuda'
assert decision['rejected']['static_only'] == ['dynamic_shape']
print(decision)


## ⚠️ 常见误区

- AI 编译器不是“把 Python 变成 C++”这么简单。
- 图优化的关键不只是算子名，而是执行路径和数据流。
- 融合不是为了让名字更高级，而是减少中间结果和搬运。
- backend 差异会直接影响优化结果。

## 相关阅读
**导语：** 如果想继续把图优化和执行后端的关系补完整，可以接着看这些页。
- [04. Triton 矩阵乘法 (GEMM) 与自动调优 (Autotune)](../03_Triton_Kernels/04_Triton_GEMM_Tutorial.ipynb)
- [05. Triton 性能调优与基准测试 (Autotune & Profiling)](../03_Triton_Kernels/05_Triton_Autotune_and_Profiling.ipynb)
- [18. CUDA Graph and JIT Compile | CUDA Graph 与 JIT 编译](../04_CUDA_and_System_Optimization/18_CUDA_Graph_and_JIT_Compile.ipynb)
---